# Notebook 06 - -valuation finale, seuil de d-cision et s-rialisation

Objectif : choisir un seuil de d-cision align- sur le co-t m-tier asym-trique, -valuer une seule fois sur test set, puis sauvegarder `models/final_model.joblib`.

## 0. Imports et configuration

In [ ]:
from pathlib import Path
import sys
import importlib.util
import pandas as pd
import joblib
from IPython.display import display

PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
MODULE_PATH = PROJECT_ROOT / "src" / "ml_phase3.py"
spec = importlib.util.spec_from_file_location("ml_phase3", MODULE_PATH)
ml_phase3 = importlib.util.module_from_spec(spec)
spec.loader.exec_module(ml_phase3)

load_dataset = ml_phase3.load_dataset
make_splits = ml_phase3.make_splits
threshold_analysis = ml_phase3.threshold_analysis
final_test_report = ml_phase3.final_test_report
save_json = ml_phase3.save_json

pd.set_option("display.max_columns", 120)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

## 1. Chargement du mod-le tun- et des splits

In [ ]:
df = load_dataset()
splits = make_splits(df)
X_val, y_val = splits["X_val"], splits["y_val"]
X_test, y_test = splits["X_test"], splits["y_test"]

tuned_path = MODELS_DIR / "tuned_model.joblib"
if not tuned_path.exists():
    raise FileNotFoundError("Ex?cuter d'abord notebooks/05_tuning.ipynb pour g?n?rer models/tuned_model.joblib")

model = joblib.load(tuned_path)
print(model)

## 2. Seuil de d-cision m-tier

Le co-t asym-trique choisi dans ce notebook est volontairement simple et explicable :
- FP = 20 unit-s de co-t, car une fausse alerte consomme des ressources d'observation.
- FN = 1000 unit-s de co-t, car rater un PHA est critique.

Le seuil retenu privil-gie F1 sous contrainte precision >= 0.40 et recall >= 0.90, puis minimise ce co-t.

In [ ]:
thresholds = threshold_analysis(model, X_val, y_val, min_precision=0.40, min_recall=0.90)
thresholds.to_csv(MODELS_DIR / "threshold_analysis.csv", index=False)
display(thresholds.head(15))

if thresholds["meets_objectives"].any():
    chosen = thresholds[thresholds["meets_objectives"]].sort_values(["f1", "business_cost"], ascending=[False, True]).iloc[0]
else:
    chosen = thresholds.sort_values(["f1", "business_cost"], ascending=[False, True]).iloc[0]

optimal_threshold = float(chosen["threshold"])
print(f"Seuil optimal retenu : {optimal_threshold:.4f}")
print(chosen[["precision", "recall", "f1", "business_cost", "fp", "fn"]])

## 3. -valuation finale sur test set

Le test set est utilis- une seule fois, apr-s la s-lection du mod-le et du seuil.

In [ ]:
test_report = final_test_report(model, X_test, y_test, optimal_threshold)
display(pd.DataFrame([test_report]))

confusion = pd.DataFrame(
    [[test_report["tn"], test_report["fp"]], [test_report["fn"], test_report["tp"]]],
    index=["R?el non-PHA", "R?el PHA"],
    columns=["Pr?dit non-PHA", "Pr?dit PHA"],
)
display(confusion)

## 4. S-rialisation du mod-le final

In [ ]:
final_artifact = {
    "pipeline": model,
    "threshold": optimal_threshold,
    "target": "is_potentially_hazardous",
    "positive_class": 1,
    "decision_rule": "predict PHA if score >= threshold",
}
joblib.dump(final_artifact, MODELS_DIR / "final_model.joblib")

metadata = {
    "threshold": optimal_threshold,
    "test_report": test_report,
    "interpretation": {
        "precision": "part des alertes PHA r?ellement PHA",
        "recall": "part des PHAs d?tect?s",
        "pr_auc": "qualit? du ranking sur classe minoritaire",
    },
}
save_json(metadata, MODELS_DIR / "final_model_metadata.json")
print("Artefacts sauvegard?s :")
print("- models/final_model.joblib")
print("- models/final_model_metadata.json")

## 5. Synth-se finale

La phase 3 compl-te contient maintenant :
- comparaison CV des mod-les et strat-gies (`04_modeling.ipynb`)
- tuning hyperparam-tres (`05_tuning.ipynb`)
- seuil de d-cision, test final et s-rialisation (`06_evaluation.ipynb`)